In [1]:
def ingresar_tanques_caracteristicas():
    nombre = input("Nombre: ")
    variable = input("Paste: ")
    if "U.S.A." in variable:
        nacion = "Estados Unidos"
    elif "Germany" in variable:
        nacion = "Alemania"
    elif "U.S.S.R." in variable:
        nacion = "Unión Soviética"
    elif "U.K." in variable:
        nacion = "Reino Unido"
    elif "Japan" in variable:
        nacion = "Japón"
    elif "China" in variable:
        nacion = "China"
    elif "France" in variable:
        nacion = "Francia"
    elif "European Nation" in variable:
        nacion = "Nación Europea"
    elif "Hybrid Nation" in variable:
        nacion = "Nación Híbrida" #determina el país

    hola = variable.split("Class ")[1].split(" Tier")[0]
    if "Light" == hola:
        clase = "Ligero"
    elif "Medium" == hola:
        clase = "Medio"
    elif "Heavy" == hola:
        clase = "Pesado"
    elif "TD" == hola:
        clase = "Destructor" #determina la clase

    if "Tech tree" in variable:
        tipo = "Árbol tecnológico"
    elif "Collector" in variable:
        tipo = "Coleccionista"
    elif "Premium" in variable:
        tipo = "Premium" #determina el tipo

    id = variable.split("DEV: ID ")[1].split("DEV: NAME")[0] #determina el id
    tier = variable.split("Tier ")[2].split(" Type")[0]#determina el tier

    #-----------------------------------------------------------------------

    caracteristicas = []
    if "Gun type Regular" in variable:
        caracteristicas.append(1)
        lista = [1, 6, 17, 66, [75, 80, 86, 92], 110, [143, 148], [212, 217, 222], 258, 270, [279, 284, 289, 294]]
    elif "Gun type Auto loader" in variable:
        caracteristicas.append(2)
        lista = [1, 6, 23, 82, [91, 96, 102, 108], 126, [159, 164], [228, 233, 238], 274, 286, [295, 300, 305, 310]]
    elif "Gun type Auto reloader" in variable:
        caracteristicas.append(3)
        lista = [1, 6, 23, 98, [107, 112, 118, 124], 142, [175, 180], [244, 249, 254], 290, 302, [311, 316, 321, 326]]

    caracteristica = 3
    variable = variable.split(" DPM ")[1].split(" ")
    for i in lista:
        caracteristica += 1
        try:
            if int(variable [i]) / int(variable[i + 2]) <= 1/3:
                caracteristicas.append(caracteristica)
        except TypeError:
            dividendo = 0
            divisor = 0
            for j in i:
                dividendo += int(variable [j])
                divisor += int(variable [j + 2])
            if dividendo / divisor <= 1/3:
                caracteristicas.append(caracteristica)

    if input("Torreta: ") != "":
        caracteristicas.append(15)
    if input("Casco: ") != "":
        caracteristicas.append(16)

    mensaje_tanque = f'INSERT INTO tanques (id, nombre , nacion, clase, tier, tipo) VALUES ({id}, "{nombre}", "{nacion}", "{clase}", "{tier}", "{tipo}");'
    mensaje_caracteristica = []

    for i in caracteristicas:
        mensaje_caracteristica.append(f'INSERT INTO tanques_caracteristicas (tanque_id, caracteristica_id) VALUES ({id}, {i});')

    return mensaje_tanque, mensaje_caracteristica


In [1]:
class tanque:
  def __init__(self, id, nombre, nacion, clase, tier, tipo, caracteristicas):
    self.id = id
    self.nombre = nombre
    self.nacion = nacion
    self.clase = clase
    self.tier = tier
    self.tipo = tipo
    self.caracteristicas = caracteristicas
  def agregar_caracteristicas(self, nombre):
    self.caracteristicas.append(nombre)
  def mostrar_todo(self):
    print(f"Id: {self.id}")
    print(f"Nombre: {self.nombre}")
    print(f"Nación: {self.nacion}")
    print(f"Clase: {self.clase}")
    print(f"Tier: {self.tier}")
    print(f"Tipo: {self.tipo}")
    print(f"Características:")
    for i in self.caracteristicas:
      print(i)

class caracteristica:
  def __init__(self, id, nombre):
    self.id = id
    self.nombre = nombre
  def mostrar_todo(self):
    print(f"Id: {self.id}")
    print(f"Nombre: {self.nombre}")

class caracteristicaDAO:
  def __init__(self):
    self.conexion = obtener_conexion()
  def obtener_todo(self):
    cursor = self.conexion.cursor()
    sql = "SELECT * FROM caracteristicas"
    cursor.execute(sql)
    lista = []
    for i in cursor.fetchall():
      caracteristica_1 = caracteristica(i[0], i[1])
      lista.append(caracteristica_1)
    self.conexion.close()
    return lista


class tanqueDAO:
  def __init__(self):
    self.conexion = obtener_conexion()
  def obtener_todo(self):
    cursor = self.conexion.cursor()
    sql = "SELECT * FROM tanques"
    cursor.execute(sql)
    variable = cursor.fetchall()
    lista = []
    for fila in variable:
      sql = "SELECT caracteristicas.nombre FROM tanques_caracteristicas JOIN caracteristicas ON caracteristicas.id = tanques_caracteristicas.caracteristica_id WHERE tanque_id = %s;"
      valores = [fila[0]]
      cursor.execute(sql, valores)
      caracteristicas = []
      for i in cursor.fetchall():
        caracteristicas.append(i[0])
      tanque_1 = tanque(fila[0], fila[1], fila[2], fila[3], fila[4], fila[5], caracteristicas)
      lista.append(tanque_1)
    self.conexion.close()
    return lista
  def insertar_tanques_web(self):
    try:
      mensaje_tanque, mensaje_caracteristica = ingresar_tanques_caracteristicas()
      cursor = self.conexion.cursor()
      cursor.execute(mensaje_tanque)
      print("Código ejecutado en MySQL:")
      for i in mensaje_caracteristica:
        cursor.execute(i)
        print(i)
      self.conexion.commit()
      self.conexion.close()
    except Exception as e:
      self.conexion.rollback()
      print("Error:", e)
  '''def insertar(self, tanque):
    cursor = self.conexion.cursor()
    sql = "INSERT INTO tanques(nombre, nacion, clase, tier, tipo) VALUES(%s, %s, %s, %s, %s)"
    valores = (
            tanque.nombre,
            tanque.nacion,
            tanque.clase,
            tanque.tier,
            tanque.tipo
        )
    cursor.execute(sql, valores)
    self.conexion.commit()
    print("Tanque insertado")'''
  '''def actualizar(self, tanque):
      cursor = self.conexion.cursor()
      sql = "UPDATE tanques SET nombre=%s, nacion=%s, clase=%s, tier=%s, tipo=%s WHERE id=%s"
      valores = (
            tanque.nombre,
            tanque.nacion,
            tanque.clase,
            tanque.tier,
            tanque.tipo,
            tanque.id
        )
      cursor.execute(sql, valores)
      self.conexion.commit()
      print("Tanque insertado")'''
  '''def eliminar(self, id):
    cursor = self.conexion.cursor()
    # primero eliminar relaciones
    sql_relaciones = "DELETE FROM tanques_caracteristicass WHERE tanque_id = %s"
    cursor.execute(sql_relaciones, (id,))
    # luego eliminar tanque
    sql_tanque = "DELETE FROM tanques WHERE id = %s"
    cursor.execute(sql_tanque, (id,))
    self.conexion.commit()
    print("Tanque eliminado")'''

In [ ]:
hola = tanqueDAO()
hola.insertar_tanques_web()

In [ ]:
lista_tanques = tanqueDAO()
lista_tanques = lista_tanques.obtener_todo()
for i in lista_tanques:    
    i.mostrar_todo()
    print("---")

In [ ]:
import os
host = input("Host: ")
user = input("Usuario: ")
password = input("Contraseña: ")
database = input("Base de datos: ")
port = int(input("Puerto: "))
os.system('cls' if os.name == 'nt' else 'clear')

In [71]:
import mysql.connector

def obtener_conexion():
    return mysql.connector.connect(
        host = host,
        user = user,
        password = password,
        database = database,
        port = port
    )

def carga_tanques():
    lista_tanques = tanqueDAO()
    lista_tanques = lista_tanques.obtener_todo()
    return lista_tanques

def carga_caracteristicas():
    lista_caracteristicas = caracteristicaDAO()
    lista_caracteristicas = lista_caracteristicas.obtener_todo()
    return lista_caracteristicas

#############estaría bueno poder filtrar también el tier, nacion, tipo, etc
def mostrar_caracteristicas():
  print("Sección características (ys: ya seleccionado):")
  for caracteristica in caracteristicas:
    if caracteristica.nombre in caracteristicas_seleccionadas:
      mensaje = f"{caracteristica.id}. {caracteristica.nombre} (ys)"
    else:
      mensaje = f"{caracteristica.id}. {caracteristica.nombre}"
    if caracteristica.id == 1:
      print("Potencia de fuego:\n\t", end = "")
      caracteres = 4
    elif caracteristica.id == 10:
      print("\nManiobrabilidad:\n\t", end = "")
      caracteres = 4
    elif caracteristica.id == 12:
      print("\nSupervivencia:\n\t", end = "")
      caracteres = 4
    if caracteres + len(mensaje) > 99:
      caracteres = 36
      print("\n\t", end = "")
    else:
      caracteres += 32
    print(mensaje, end = (32 - len(mensaje)) * " ")
  print()

def mostrar_tanques():
  print("Sección tanques:\n\t", end = "")
  if caracteristicas_seleccionadas or tanques_seleccionados:
    lista = tanques_seleccionados
  else:
    lista = tanques
  caracteres = 4
  for i in range(len(lista)):
    mensaje = f"T{i + 1}. {lista[i].nombre}"
    if caracteres + len(mensaje) > 87:
      caracteres = 32
      print("\n\t", end = "")
    else:
      caracteres += 28
    print(mensaje, end = (26 - len(mensaje)) * " ")
  print()

def eleccion():
  respuesta = input("Elección: ")
  return respuesta


  





   ######### habría que separar esta función en tres, todavía no se si quedarme con lo anterior
   #porque el anterior no ocupa espacio en el programa principal, pero pide returns innecesarios
  """def eleccion:
  return respuesta
  def mostrar tanque
  def filtrar
  while:
  if eleccion() == "":
  mostrar tanque()

  else:
  filtrar()
  respuesta = input()
  try:
    if "T" in respuesta or "t" in respuesta:
      mostrar_tanque(respuesta)
      return caracteristicas_seleccionadas, tanques_seleccionados
    else:
      caracteristicas_seleccionadas = seleccionar_caracteristica(respuesta)
      tanques_seleccionados = seleccionar_tanques()
      return caracteristicas_seleccionadas, tanques_seleccionados
  except:
    return caracteristicas_seleccionadas, tanques_seleccionados
    '''if respuesta == "":
      break   ################### hay que sumar alguna manera de parar el bucle
    else:
      print("Ingresa un valor válido")
      eleccion(tanques, tanques_seleccionados, caracteristicas, caracteristicas_seleccionadas)'''"""
    
def seleccionar_caracteristica():
  print(f"Elección: {respuesta}. {caracteristicas[int(respuesta) - 1].nombre}")
  if caracteristicas[int(respuesta) - 1].nombre in caracteristicas_seleccionadas:
    caracteristicas_seleccionadas.remove(caracteristicas[int(respuesta) - 1].nombre)
  else:
    caracteristicas_seleccionadas.append(caracteristicas[int(respuesta) - 1].nombre)
  return caracteristicas_seleccionadas

def seleccionar_tanques():
  tanques_seleccionados = tanques[:]
  for tanque in tanques:
    for caracteristica in caracteristicas_seleccionadas:
      if caracteristica not in tanque.caracteristicas:
        tanques_seleccionados.remove(tanque)
        break
  return tanques_seleccionados

def mostrar_tanque():
  respuesta = int(respuesta.split("T")[1]) - 1
  if tanques_seleccionados:
    lista = tanques_seleccionados
  else:
    lista = tanques
  lista[respuesta].mostrar_todo()



  



"""def seleccionar_tanques_caracteristicas(tanques, caracteristicas, caracteristicas_seleccionadas):#filtros, filtros_seleccionados, tanques ,tanques_filtrados, filtros_lista):
  respuesta = input("Elección: ")
  try:
    respuesta = int(respuesta)
    if caracteristicas[respuesta - 1] not in caracteristicas_seleccionadas:
      caracteristicas_seleccionadas.append(caracteristicas[respuesta - 1])
    else:
      caracteristicas_seleccionadas.remove(caracteristicas[respuesta - 1])
  except:
    respuesta = respuesta.upper()
    respuesta = respuesta.split("T")[1]
    for i in tanques:
#mostrar todo del tanque
"""
"""
  except:
    try:
      respuesta = int(respuesta.upper().split("T")[1])
      if tanques_filtrados:
        clase = tanques_filtrados[respuesta - 1]
      else:
        clase = tanques[respuesta - 1]
      print(f"Tanque : {clase.nombre}")
      for a, b in clase.informacion.items():
        print(f"{a}: {b}")
      print(f"Características:")
      for i in clase.caracteristicas:
        print(i)
      input()
    except:
      print("Elige una de las opciones correspondientes")
      filtros_seleccionados = preguntar(filtros, filtros_seleccionados, tanques ,tanques_filtrados)
  return filtros_seleccionados"""


'\n  except:\n    try:\n      respuesta = int(respuesta.upper().split("T")[1])\n      if tanques_filtrados:\n        clase = tanques_filtrados[respuesta - 1]\n      else:\n        clase = tanques[respuesta - 1]\n      print(f"Tanque : {clase.nombre}")\n      for a, b in clase.informacion.items():\n        print(f"{a}: {b}")\n      print(f"Características:")\n      for i in clase.caracteristicas:\n        print(i)\n      input()\n    except:\n      print("Elige una de las opciones correspondientes")\n      filtros_seleccionados = preguntar(filtros, filtros_seleccionados, tanques ,tanques_filtrados)\n  return filtros_seleccionados'

In [72]:
caracteristicas_seleccionadas = []
tanques_seleccionados = []
caracteristicas = carga_caracteristicas()
tanques = carga_tanques()
while True:
  mostrar_caracteristicas()
  mostrar_tanques()
  respuesta = eleccion()
  try:
    if "T" in respuesta or "t" in respuesta:
      mostrar_tanque()
    else:
      caracteristicas_seleccionadas = seleccionar_caracteristica()
      tanques_seleccionados = seleccionar_tanques()
  except:
    if respuesta == "" or respuesta == None:
      break
    else:
      print("Ingrese un valor válido")###todavía hay que corregir cuando se quedan sin tanques

Sección características (ys: ya seleccionado):
Potencia de fuego:
	1. Cañón convencional           2. Cargador automático          3. Recargador automático        
	4. Daño por minuto              5. Daño por disparo             6. Penetración                  
	7. Tiempo de apuntado           8. Dispersión                   9. Depresión del cañón          
Maniobrabilidad:
	10. Velocidad máxima            11. Relación potencia/peso      
Supervivencia:
	12. Puntos de vida              13. Alcance de visión           14. Camuflaje                   
	15. Blindaje de la torreta      16. Blindaje del casco          
Sección tanques:
	T1. WZ-132A               T2. Sheridan              T3. Kanonenjagdpanzer     
	T4. Rhm. Pzw.             T5. FCM 50 t              
Elección: 10. Velocidad máxima
Sección características (ys: ya seleccionado):
Potencia de fuego:
	1. Cañón convencional           2. Cargador automático          3. Recargador automático        
	4. Daño por minuto             